# nuPlan Mini Transfer Benchmark for FM + CBF

这个 notebook 按你的论文实验思路组织：

- 场景筛选：`following_lane_without_lead` 近似 `lane_following`，`starting_left_turn` 近似 `left_turn`
- 方法对比：`FM+CBF`、`Pure FM`、`IDM-proxy`
- 指标：`time_to_goal`、`collision_rate`、`goal_reached_rate`、`min_inter_agent_dist`、`road_boundary_violations`

当前环境里没有安装 `nuplan` SDK，所以本 notebook 采用两层方案：

1. 先用 `sqlite3` 直接读 `nuPlan mini .db`，完成场景筛选、帧提取和离线 benchmark。
2. 预留 `official_idm` / `official_pdm_closed` 接口。等你安装 `nuplan-devkit` 后，把 planner hook 接进去即可。

当前的两个近似假设：

- `road corridor` 先用 expert future ego trajectory 代理，用于道路边界约束和 `road_boundary_violations` 统计。
- surrounding agents 使用 log-replay；ego 由 planner 闭环 rollout。

这很适合你说的阶段一：先把 pipeline 跑通。阶段二再把训练和官方 baseline 接完整。

补充：核心 benchmark 逻辑现在也已经抽到包模块 `dmpc_fm_cbf.nuplan_benchmark`，这个 notebook 主要保留交互式配置和可视化。

In [ ]:
from __future__ import annotations

import math
import random
import sqlite3
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path('/new_world/cockatiel/ra').resolve()
PROJECT_ROOT = ROOT / 'safeflow-nuplan-transfer' / 'dmpc_fm_cbf'
NOTEBOOK_ROOT = PROJECT_ROOT / 'notebooks'
DATA_ROOT = ROOT / 'data' / 'cache' / 'mini'
MAP_ROOT = ROOT / 'maps'
OUTPUT_DIR = NOTEBOOK_ROOT / 'cache' / 'nuplan_transfer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fmtorch.models.simple1d_unet import SimpleUNet1D
from dmpc_fm_cbf.canonical import build_can_tf_5d
from dmpc_fm_cbf.sampler_5h import piecewise_sample_fm_5ch_with_cbf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)
print('DATA_ROOT =', DATA_ROOT)
print('MAP_ROOT  =', MAP_ROOT)


In [ ]:
@dataclass
class BenchmarkConfig:
    seed: int = 7
    scenarios_per_type: int = 8
    episode_steps: int = 40
    horizon_steps: int = 16
    replay_dt_fallback: float = 0.1
    max_agents: int = 8
    goal_tol_m: float = 8.0
    safe_radius_m: float = 3.0
    road_half_width_m: float = 4.5
    road_margin_m: float = 0.5
    idm_desired_speed_mps: float = 8.0
    fm_noise_scale: float = 1.0
    fm_num_segments: int = 20
    fm_tau0: float = 0.50
    fm_tau1: float = 0.90
    fm_goal_guidance_weight: float = 1.20
    fm_start_guidance_weight: float = 0.40
    cbf_extra_margin_m: float = 0.25
    cbf_alpha: float = 8.0
    cbf_max_corr_norm: float = 2.0
    methods: Tuple[str, ...] = ('fm_cbf', 'pure_fm', 'idm_proxy')
    scenario_type_map: Dict[str, str] = None
    model_ckpt: Path = NOTEBOOK_ROOT / 'cache' / 'model_vel.pt'

    def __post_init__(self):
        if self.scenario_type_map is None:
            self.scenario_type_map = {
                'lane_following': 'following_lane_without_lead',
                'left_turn': 'starting_left_turn',
            }


@dataclass
class ScenarioRecord:
    db_path: Path
    scene_token: bytes
    anchor_lidar_pc_token: bytes
    goal_ego_pose_token: Optional[bytes]
    scenario_tag: str
    paper_bucket: str
    scene_name: str
    location: str
    map_version: str


@dataclass
class EpisodeMetrics:
    method: str
    paper_bucket: str
    scenario_tag: str
    scene_name: str
    time_to_goal_s: float
    goal_reached: int
    collision: int
    min_inter_agent_dist_m: float
    road_boundary_violations: int
    final_dist_to_goal_m: float
    num_steps: int


CFG = BenchmarkConfig()
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

CFG


In [ ]:
def blob_hex(blob: Optional[bytes]) -> Optional[str]:
    return None if blob is None else blob.hex()


def wrap_pi(angle: float) -> float:
    return float((angle + np.pi) % (2 * np.pi) - np.pi)


def yaw_from_quaternion(qw: float, qx: float, qy: float, qz: float) -> float:
    siny_cosp = 2.0 * (qw * qz + qx * qy)
    cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
    return float(np.arctan2(siny_cosp, cosy_cosp))


def polyline_distance(point_xy: np.ndarray, polyline_xy: np.ndarray) -> float:
    p = np.asarray(point_xy, dtype=np.float64).reshape(2)
    pts = np.asarray(polyline_xy, dtype=np.float64)
    if len(pts) == 0:
        return float('inf')
    if len(pts) == 1:
        return float(np.linalg.norm(p - pts[0]))

    best = float('inf')
    for a, b in zip(pts[:-1], pts[1:]):
        ab = b - a
        denom = float(np.dot(ab, ab))
        if denom < 1e-9:
            d = np.linalg.norm(p - a)
        else:
            t = np.clip(np.dot(p - a, ab) / denom, 0.0, 1.0)
            proj = a + t * ab
            d = np.linalg.norm(p - proj)
        best = min(best, float(d))
    return best


def nearest_agent_distance(ego_xy: np.ndarray, agents: List[dict]) -> float:
    if not agents:
        return float('inf')
    dists = [float(np.linalg.norm(ego_xy - a['xy'])) for a in agents]
    return float(min(dists))


def select_nearest_agents(ego_xy: np.ndarray, agents: List[dict], k: int) -> List[dict]:
    agents = sorted(agents, key=lambda a: np.linalg.norm(a['xy'] - ego_xy))
    return agents[:k]


def make_runtime_state(frame: dict, prev_state: Optional[np.ndarray] = None) -> np.ndarray:
    x = float(frame['ego_x'])
    y = float(frame['ego_y'])
    theta = float(frame['ego_yaw'])
    v = float(math.hypot(frame['ego_vx'], frame['ego_vy']))
    yaw_rate = 0.0 if prev_state is None else wrap_pi(theta - float(prev_state[2])) / max(frame['dt'], 1e-3)
    return np.array([x, y, theta, v, yaw_rate], dtype=np.float32)


def rollout_point_mass_to_target(state: np.ndarray, target_xy: np.ndarray, dt: float, speed_cap: float) -> np.ndarray:
    xy = state[:2].astype(np.float64)
    target = np.asarray(target_xy, dtype=np.float64).reshape(2)
    delta = target - xy
    dist = float(np.linalg.norm(delta))
    if dist < 1e-6:
        return state.copy()
    direction = delta / dist
    speed = min(speed_cap, dist / max(dt, 1e-3))
    next_xy = xy + direction * speed * dt
    theta = float(np.arctan2(direction[1], direction[0]))
    yaw_rate = wrap_pi(theta - float(state[2])) / max(dt, 1e-3)
    return np.array([next_xy[0], next_xy[1], theta, speed, yaw_rate], dtype=np.float32)


def load_model_or_none(cfg: BenchmarkConfig):
    if not cfg.model_ckpt.exists():
        print(f'[WARN] checkpoint not found: {cfg.model_ckpt}')
        print('[WARN] FM planners will fail until you set CFG.model_ckpt to your real checkpoint.')
        return None

    model = SimpleUNet1D(cond_dim=5, hidden_dim=128, in_channels=5, out_channels=5)
    ckpt = torch.load(cfg.model_ckpt, map_location=DEVICE)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE).eval()
    print(f'[OK] model loaded from {cfg.model_ckpt}')
    return model


MODEL_5CH = load_model_or_none(CFG)


In [ ]:
def discover_scenarios(cfg: BenchmarkConfig) -> List[ScenarioRecord]:
    records: List[ScenarioRecord] = []
    db_files = sorted(DATA_ROOT.glob('*.db'))
    print(f'found {len(db_files)} mini db files')

    for db_path in db_files:
        con = sqlite3.connect(str(db_path))
        cur = con.cursor()
        log_row = cur.execute('SELECT location, map_version FROM log LIMIT 1').fetchone()
        location, map_version = log_row if log_row is not None else ('unknown', 'unknown')

        for paper_bucket, nuplan_tag in cfg.scenario_type_map.items():
            rows = cur.execute(
                '''
                SELECT DISTINCT lp.scene_token, st.lidar_pc_token, s.goal_ego_pose_token, s.name, st.type
                FROM scenario_tag st
                JOIN lidar_pc lp ON st.lidar_pc_token = lp.token
                JOIN scene s ON lp.scene_token = s.token
                WHERE st.type = ?
                ORDER BY s.name
                ''',
                (nuplan_tag,),
            ).fetchall()

            for scene_token, anchor_token, goal_pose_token, scene_name, scenario_tag in rows:
                records.append(
                    ScenarioRecord(
                        db_path=db_path,
                        scene_token=scene_token,
                        anchor_lidar_pc_token=anchor_token,
                        goal_ego_pose_token=goal_pose_token,
                        scenario_tag=scenario_tag,
                        paper_bucket=paper_bucket,
                        scene_name=scene_name,
                        location=location,
                        map_version=map_version,
                    )
                )
        con.close()
    return records


def sample_balanced_scenarios(records: List[ScenarioRecord], cfg: BenchmarkConfig) -> List[ScenarioRecord]:
    chosen: List[ScenarioRecord] = []
    rng = random.Random(cfg.seed)
    for bucket in cfg.scenario_type_map.keys():
        bucket_records = [r for r in records if r.paper_bucket == bucket]
        rng.shuffle(bucket_records)
        chosen.extend(bucket_records[: cfg.scenarios_per_type])
    return chosen


ALL_SCENARIOS = discover_scenarios(CFG)
SELECTED_SCENARIOS = sample_balanced_scenarios(ALL_SCENARIOS, CFG)
pd.DataFrame([
    {
        'db': s.db_path.name,
        'scene_name': s.scene_name,
        'paper_bucket': s.paper_bucket,
        'scenario_tag': s.scenario_tag,
        'location': s.location,
    }
    for s in SELECTED_SCENARIOS
]).head(20)


In [ ]:
def load_scene_bundle(record: ScenarioRecord, cfg: BenchmarkConfig) -> dict:
    con = sqlite3.connect(str(record.db_path))
    cur = con.cursor()

    frame_rows = cur.execute(
        '''
        SELECT lp.token, lp.timestamp,
               ep.x, ep.y, ep.qw, ep.qx, ep.qy, ep.qz,
               ep.vx, ep.vy
        FROM lidar_pc lp
        JOIN ego_pose ep ON lp.ego_pose_token = ep.token
        WHERE lp.scene_token = ?
        ORDER BY lp.timestamp
        ''',
        (record.scene_token,),
    ).fetchall()

    goal_xy = None
    if record.goal_ego_pose_token is not None:
        row = cur.execute('SELECT x, y FROM ego_pose WHERE token = ?', (record.goal_ego_pose_token,)).fetchone()
        if row is not None:
            goal_xy = np.array(row, dtype=np.float32)

    frames = []
    last_ts = None
    for token, ts, x, y, qw, qx, qy, qz, vx, vy in frame_rows:
        dt = cfg.replay_dt_fallback if last_ts is None else max((ts - last_ts) * 1e-6, 1e-3)
        last_ts = ts
        yaw = yaw_from_quaternion(qw, qx, qy, qz)

        agents = []
        box_rows = cur.execute(
            '''
            SELECT lb.track_token, lb.x, lb.y, lb.vx, lb.vy, lb.yaw, lb.width, lb.length, c.name
            FROM lidar_box lb
            JOIN track t ON lb.track_token = t.token
            JOIN category c ON t.category_token = c.token
            WHERE lb.lidar_pc_token = ? AND c.name = 'vehicle'
            ''',
            (token,),
        ).fetchall()

        for track_token, ax, ay, avx, avy, ayaw, width, length, category in box_rows:
            agents.append(
                {
                    'track_token': track_token,
                    'xy': np.array([ax, ay], dtype=np.float32),
                    'vx': float(avx),
                    'vy': float(avy),
                    'yaw': float(ayaw),
                    'width': float(width),
                    'length': float(length),
                    'category': category,
                }
            )

        frames.append(
            {
                'lidar_pc_token': token,
                'timestamp': ts,
                'dt': dt,
                'ego_x': float(x),
                'ego_y': float(y),
                'ego_yaw': yaw,
                'ego_vx': float(vx),
                'ego_vy': float(vy),
                'agents': agents,
            }
        )

    con.close()
    anchor_index = next(i for i, f in enumerate(frames) if f['lidar_pc_token'] == record.anchor_lidar_pc_token)
    if goal_xy is None:
        goal_xy = np.array([frames[-1]['ego_x'], frames[-1]['ego_y']], dtype=np.float32)

    corridor = np.array([[f['ego_x'], f['ego_y']] for f in frames[anchor_index:]], dtype=np.float32)
    return {
        'record': record,
        'frames': frames,
        'anchor_index': anchor_index,
        'goal_xy': goal_xy,
        'corridor_xy': corridor,
    }


SCENE_CACHE: Dict[Tuple[str, str], dict] = {}

def get_scene_bundle(record: ScenarioRecord, cfg: BenchmarkConfig) -> dict:
    key = (str(record.db_path), blob_hex(record.anchor_lidar_pc_token))
    if key not in SCENE_CACHE:
        SCENE_CACHE[key] = load_scene_bundle(record, cfg)
    return SCENE_CACHE[key]


sample_bundle = get_scene_bundle(SELECTED_SCENARIOS[0], CFG)
print(sample_bundle['record'].scene_name, sample_bundle['record'].paper_bucket)
print('frames =', len(sample_bundle['frames']), 'anchor_index =', sample_bundle['anchor_index'])
print('goal_xy =', sample_bundle['goal_xy'])


In [ ]:
def fm_sample_path_world(
    state: np.ndarray,
    goal_xy: np.ndarray,
    agents: List[dict],
    cfg: BenchmarkConfig,
    model,
    use_cbf: bool,
) -> np.ndarray:
    if model is None:
        raise FileNotFoundError('MODEL_5CH is None. Set CFG.model_ckpt to your trained checkpoint path first.')

    start_xy = state[:2].astype(np.float32)
    goal_xy = np.asarray(goal_xy, dtype=np.float32).reshape(2)
    tf = build_can_tf_5d(start_xy, goal_xy)
    goal_dist = float(tf['dist'])
    theta_can = wrap_pi(float(state[2]) - float(tf['phi']))

    cond = np.array(
        [goal_dist, float(state[3]), math.cos(theta_can), math.sin(theta_can), float(state[4])],
        dtype=np.float32,
    )

    ellipses = []
    for agent in select_nearest_agents(start_xy, agents, cfg.max_agents):
        c_can = tf['w2c'](agent['xy'].reshape(1, 2))[0].astype(np.float32)
        radius = 0.5 * max(agent['width'], agent['length']) + cfg.safe_radius_m * 0.25
        ellipses.append({'center': c_can, 'radius': float(radius)})

    x_can, _, _ = piecewise_sample_fm_5ch_with_cbf(
        model_5ch=model,
        T_steps=cfg.horizon_steps,
        device=DEVICE,
        num_segments=cfg.fm_num_segments,
        noise_scale=cfg.fm_noise_scale,
        cond=cond,
        start_xy_norm=np.array([0.0, 0.0], dtype=np.float32),
        lock_start=True,
        goal_xy_norm=np.array([goal_dist, 0.0], dtype=np.float32),
        start_guidance_weight=cfg.fm_start_guidance_weight,
        goal_guidance_weight=cfg.fm_goal_guidance_weight,
        use_cbf=use_cbf,
        ellipses=ellipses,
        tau0=cfg.fm_tau0,
        tau1=cfg.fm_tau1,
        cbf_kwargs={
            'passes': 8,
            'extra_margin': cfg.cbf_extra_margin_m,
            'alpha': cfg.cbf_alpha,
            'max_corr_norm': cfg.cbf_max_corr_norm,
        },
    )

    path_can = np.asarray(x_can[:2, :].T, dtype=np.float32)
    path_world = tf['c2w'](path_can).astype(np.float32)
    return path_world


def idm_proxy_step(state: np.ndarray, corridor_xy: np.ndarray, agents: List[dict], dt: float, cfg: BenchmarkConfig) -> np.ndarray:
    ego_xy = state[:2].astype(np.float32)
    if len(corridor_xy) == 0:
        return state.copy()

    dists = np.linalg.norm(corridor_xy - ego_xy[None, :], axis=1)
    idx = int(np.argmin(dists))
    lookahead_idx = min(idx + 5, len(corridor_xy) - 1)
    target_xy = corridor_xy[lookahead_idx]

    desired_speed = cfg.idm_desired_speed_mps
    nearest = select_nearest_agents(ego_xy, agents, 1)
    if nearest:
        lead_dist = float(np.linalg.norm(nearest[0]['xy'] - ego_xy))
        if lead_dist < 20.0:
            desired_speed = min(desired_speed, max(0.0, lead_dist - cfg.safe_radius_m))

    return rollout_point_mass_to_target(state, target_xy, dt=dt, speed_cap=desired_speed)


def choose_path_target(path_xy: np.ndarray, state_xy: np.ndarray) -> np.ndarray:
    if len(path_xy) == 0:
        return state_xy.copy()
    for pt in path_xy[1:]:
        if np.linalg.norm(pt - state_xy) > 0.5:
            return pt
    return path_xy[-1]


def run_episode(bundle: dict, method: str, cfg: BenchmarkConfig, model=None) -> Tuple[EpisodeMetrics, pd.DataFrame]:
    frames = bundle['frames']
    anchor_idx = bundle['anchor_index']
    goal_xy = bundle['goal_xy'].astype(np.float32)
    corridor_xy = bundle['corridor_xy'].astype(np.float32)
    record = bundle['record']

    state = make_runtime_state(frames[anchor_idx])
    traces = []
    min_dist = float('inf')
    road_violations = 0
    collision = 0
    reached = 0
    time_to_goal = cfg.episode_steps * cfg.replay_dt_fallback

    for step in range(cfg.episode_steps):
        frame_idx = min(anchor_idx + step, len(frames) - 1)
        frame = frames[frame_idx]
        dt = float(frame['dt'])
        agents = select_nearest_agents(state[:2], frame['agents'], cfg.max_agents)

        if method == 'fm_cbf':
            path_xy = fm_sample_path_world(state, goal_xy, agents, cfg, model, use_cbf=True)
            target_xy = choose_path_target(path_xy, state[:2])
            state = rollout_point_mass_to_target(state, target_xy, dt=dt, speed_cap=cfg.idm_desired_speed_mps)
        elif method == 'pure_fm':
            path_xy = fm_sample_path_world(state, goal_xy, agents, cfg, model, use_cbf=False)
            target_xy = choose_path_target(path_xy, state[:2])
            state = rollout_point_mass_to_target(state, target_xy, dt=dt, speed_cap=cfg.idm_desired_speed_mps)
        elif method == 'idm_proxy':
            path_xy = None
            state = idm_proxy_step(state, corridor_xy, agents, dt=dt, cfg=cfg)
        elif method in {'official_idm', 'official_pdm_closed'}:
            raise NotImplementedError(
                'nuplan SDK is not installed in this environment. Add official planner hook after installing nuplan-devkit.'
            )
        else:
            raise ValueError(f'unknown method: {method}')

        dist_goal = float(np.linalg.norm(state[:2] - goal_xy))
        agent_dist = nearest_agent_distance(state[:2], agents)
        min_dist = min(min_dist, agent_dist)
        road_err = polyline_distance(state[:2], corridor_xy)
        road_violations += int(road_err > (cfg.road_half_width_m + cfg.road_margin_m))
        if agent_dist < cfg.safe_radius_m:
            collision = 1

        traces.append(
            {
                'step': step,
                'x': float(state[0]),
                'y': float(state[1]),
                'theta': float(state[2]),
                'speed': float(state[3]),
                'dist_to_goal_m': dist_goal,
                'nearest_agent_dist_m': agent_dist,
                'road_err_m': road_err,
            }
        )

        if dist_goal < cfg.goal_tol_m:
            reached = 1
            time_to_goal = float(step + 1) * dt
            break

    trace_df = pd.DataFrame(traces)
    final_dist = float(trace_df['dist_to_goal_m'].iloc[-1]) if len(trace_df) else float(np.linalg.norm(state[:2] - goal_xy))

    metrics = EpisodeMetrics(
        method=method,
        paper_bucket=record.paper_bucket,
        scenario_tag=record.scenario_tag,
        scene_name=record.scene_name,
        time_to_goal_s=float(time_to_goal),
        goal_reached=int(reached),
        collision=int(collision),
        min_inter_agent_dist_m=float(min_dist),
        road_boundary_violations=int(road_violations),
        final_dist_to_goal_m=float(final_dist),
        num_steps=int(len(trace_df)),
    )
    return metrics, trace_df


In [ ]:
RUN_METHODS = CFG.methods
RUN_SCENARIOS = SELECTED_SCENARIOS

all_metrics = []
all_traces = []

for scenario in RUN_SCENARIOS:
    bundle = get_scene_bundle(scenario, CFG)
    for method in RUN_METHODS:
        try:
            metrics, trace_df = run_episode(bundle, method=method, cfg=CFG, model=MODEL_5CH)
            all_metrics.append(asdict(metrics))
            trace_df = trace_df.copy()
            trace_df['method'] = method
            trace_df['scene_name'] = scenario.scene_name
            trace_df['paper_bucket'] = scenario.paper_bucket
            all_traces.append(trace_df)
            print(f"[OK] {scenario.scene_name:<12} {scenario.paper_bucket:<14} {method:<10} goal={metrics.goal_reached} collision={metrics.collision}")
        except Exception as exc:
            print(f"[FAIL] {scenario.scene_name} {method}: {exc}")

metrics_df = pd.DataFrame(all_metrics)
traces_df = pd.concat(all_traces, ignore_index=True) if all_traces else pd.DataFrame()

metrics_csv = OUTPUT_DIR / 'metrics.csv'
traces_csv = OUTPUT_DIR / 'traces.csv'
metrics_df.to_csv(metrics_csv, index=False)
if len(traces_df):
    traces_df.to_csv(traces_csv, index=False)

print('saved:', metrics_csv)
print('saved:', traces_csv)
metrics_df.head()


In [ ]:
if len(metrics_df) == 0:
    raise RuntimeError('No benchmark result produced. Check model path or scenario selection.')

summary_df = (
    metrics_df
    .groupby(['paper_bucket', 'method'], as_index=False)
    .agg(
        time_to_goal_s=('time_to_goal_s', 'mean'),
        collision_rate=('collision', 'mean'),
        goal_reached_rate=('goal_reached', 'mean'),
        min_inter_agent_dist_m=('min_inter_agent_dist_m', 'mean'),
        road_boundary_violations=('road_boundary_violations', 'mean'),
        final_dist_to_goal_m=('final_dist_to_goal_m', 'mean'),
    )
    .sort_values(['paper_bucket', 'method'])
)

display(summary_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in [
    (axes[0], 'collision_rate', 'Collision Rate ↓'),
    (axes[1], 'goal_reached_rate', 'Goal Reached Rate ↑'),
    (axes[2], 'road_boundary_violations', 'Road Violations ↓'),
]:
    pivot = summary_df.pivot(index='paper_bucket', columns='method', values=metric)
    pivot.plot(kind='bar', ax=ax, rot=0, title=title)
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'summary_barplots.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
# 可视化一个 episode
scene_name = metrics_df.iloc[0]['scene_name']
method_name = metrics_df.iloc[0]['method']
record = next(r for r in RUN_SCENARIOS if r.scene_name == scene_name)
bundle = get_scene_bundle(record, CFG)
trace_df = traces_df[(traces_df['scene_name'] == scene_name) & (traces_df['method'] == method_name)].copy()

plt.figure(figsize=(6, 6))
corridor = bundle['corridor_xy']
plt.plot(corridor[:, 0], corridor[:, 1], '--', lw=2, label='expert corridor')
plt.plot(trace_df['x'], trace_df['y'], lw=2, label=f'{method_name} ego rollout')
plt.scatter([bundle['goal_xy'][0]], [bundle['goal_xy'][1]], marker='*', s=180, label='goal')
plt.axis('equal')
plt.title(f'{scene_name} | {method_name}')
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


## 下一步怎么补成论文版

1. 把 `CFG.model_ckpt` 指到你真实训练好的 `SimpleUNet1D` checkpoint。
2. 如果阶段二要在 nuPlan mini 上重训，把训练输出也统一存到 `notebooks/cache/` 或单独的 `checkpoints/` 目录。
3. 安装 `nuplan-devkit` 后补两个 planner hook：
   - `official_idm`
   - `official_pdm_closed`
4. 如果你要做你提到的消融：
   - 改 `CFG.fm_tau0 / CFG.fm_tau1`
   - 改 `cfg.cbf_extra_margin_m`
   - 改 candidate 数 `K` 时，把 `fm_sample_path_world` 外面包一层多样本采样与轨迹打分
5. 当前 road CBF 用的是 `expert future corridor proxy`。如果你后面把 map adapter 接好，可以把 corridor 替换成真正的 lane centerline / route lane sequence。
6. 官方 baseline 命令生成已经封装到 `dmpc_fm_cbf.nuplan_sdk_adapter` 和 `scripts/print_nuplan_sim_command.py`。安装好 `nuplan-devkit` / `tuplan_garage` 后，可以直接生成 `official_idm` / `official_pdm_closed` 的标准 `run_simulation.py` 命令。
7. 安装与运行步骤见：[NUPLAN_BASELINE_SETUP.md](/new_world/cockatiel/ra/safeflow-nuplan-transfer/dmpc_fm_cbf/docs/NUPLAN_BASELINE_SETUP.md)